In [0]:
from pyspark.sql.functions import *

In [0]:
data1=[(100,"Raj",None,1,"01-04-23",50000),
       (200,"Joanne",100,1,"01-04-23",4000),(200,"Joanne",100,1,"13-04-23",4500),(200,"Joanne",100,1,"14-04-23",4020)]
schema1=["EmpId","EmpName","Mgrid","deptid","salarydt","salary"]
df_salary=spark.createDataFrame(data1,schema1)
display(df_salary)
#department dataframe
data2=[(1,"IT"),
       (2,"HR")]
schema2=["deptid","deptname"]
df_dept=spark.createDataFrame(data2,schema2)
display(df_dept)

EmpId,EmpName,Mgrid,deptid,salarydt,salary
100,Raj,null,1,01-04-23,50000
200,Joanne,100,1,01-04-23,4000
200,Joanne,100,1,13-04-23,4500
200,Joanne,100,1,14-04-23,4020


deptid,deptname
1,IT
2,HR


In [0]:
df = df_salary.join(df_dept,df_salary.deptid==df_dept.deptid,"inner").drop(df_dept.deptid)
df.display()

EmpId,EmpName,Mgrid,deptid,salarydt,salary,deptname
100,Raj,null,1,01-04-23,50000,IT
200,Joanne,100,1,01-04-23,4000,IT
200,Joanne,100,1,13-04-23,4500,IT
200,Joanne,100,1,14-04-23,4020,IT


In [0]:
df1 = df.alias("a").join(df.alias("b"), col("a.Mgrid") == col("b.EmpId") , "Left").select(col("a.*"), col("b.Empname").alias("ManagerName"))
df1.display()

EmpId,EmpName,Mgrid,deptid,salarydt,salary,deptname,ManagerName
100,Raj,null,1,01-04-23,50000,IT,null
200,Joanne,100,1,01-04-23,4000,IT,Raj
200,Joanne,100,1,13-04-23,4500,IT,Raj
200,Joanne,100,1,14-04-23,4020,IT,Raj


In [0]:
df2 = df1.withColumn("salarydt", to_date(col("salarydt"),"dd-MM-yy"))
df2.display()

EmpId,EmpName,Mgrid,deptid,salarydt,salary,deptname,ManagerName
100,Raj,null,1,2023-04-01,50000,IT,null
200,Joanne,100,1,2023-04-01,4000,IT,Raj
200,Joanne,100,1,2023-04-13,4500,IT,Raj
200,Joanne,100,1,2023-04-14,4020,IT,Raj


In [0]:
df3 = df2.groupBy("deptname","ManagerName","EmpName",year("salarydt").alias("year"),date_format("salarydt","MMMM").alias("month")).agg(sum("salary").alias("sum(salary)"))
df3.display()

deptname,ManagerName,EmpName,year,month,sum(salary)
IT,null,Raj,2023,April,50000
IT,Raj,Joanne,2023,April,12520
